# 01 - 数据加载与缓存

本 notebook 演示如何使用 `utilities/` 下的加载器获取数据：
- **Binance aggTrades**（含 taker 聚合信息：first_trade_id, last_trade_id）
- **Binance klines**（1m OHLCV）
- **Lake-API orderbook**（历史深度数据）

所有数据按 `data/cache/{symbol}/trades|klines|orderbook/{date}.parquet` 自动分区缓存。

In [ ]:
import sys
sys.path.insert(0, '..')

from datetime import date, timedelta
import pandas as pd

from utilities.binance_loader import load_agg_trades, load_klines, resample_trades_to_ohlcv
from utilities.lakeapi_loader import load_lakeapi_orderbook
from utilities.paths import DataPaths

# 配置
SYMBOL = 'SOL/USDT'
DAYS   = 3
print(f'数据根目录: {DataPaths.data}')

## 1. Binance aggTrades

aggTrades 与 lake-api trades 的关键区别：
- `agg_trade_id`: 聚合 trade ID
- `first_trade_id` / `last_trade_id`: 同一 taker order 的首末撮合 ID
- `n_trades_in_agg`: 聚合了多少笔原始撮合（= last - first + 1）

这些字段让我们能准确还原 taker 吃了多少档流动性。

In [ ]:
# 加载 aggTrades（首次会从 Binance 拉取并缓存）
trades = load_agg_trades(symbol=SYMBOL, days=DAYS)
print(f'Total aggTrades: {len(trades):,}')
print(f'时间范围: {trades["timestamp"].min()} ~ {trades["timestamp"].max()}')
trades.head(10)

In [ ]:
# taker 聚合分布
print('n_trades_in_agg 分布（每条 aggTrade 聚合了多少笔撮合）：')
trades['n_trades_in_agg'].describe()

In [ ]:
# Resample 为 1min OHLCV
ohlcv_1m = resample_trades_to_ohlcv(trades, freq='1min')
print(f'1min bars: {len(ohlcv_1m)}')
ohlcv_1m.head()

## 2. Binance Klines (REST API)

作为 aggTrades resample 的对照。

In [ ]:
klines = load_klines(symbol=SYMBOL, timeframe='1m', days=DAYS)
print(f'Klines 1m bars: {len(klines)}')
klines.head()

## 3. Lake-API Orderbook（历史深度）

注意：sample 数据仅覆盖 BTC-USDT 2022-10-01~03。
若需 SOL/USDT 的 orderbook，需要 Tardis 付费 key。

In [ ]:
# Lake-API orderbook (BTC-USDT sample)
try:
    book = load_lakeapi_orderbook(days=1, symbol='BTC-USDT')
    print(f'Orderbook snapshots: {len(book)}')
    book.head()
except Exception as e:
    print(f'Lake-API not available: {e}')

## 4. 数据缓存结构

数据自动按以下目录结构缓存：
```
data/cache/
  SOL_USDT/
    trades/
      2026-01-31.parquet
      2026-02-01.parquet
    klines/
      1m_2026-01-31.parquet
```

In [ ]:
from pathlib import Path
cache_dir = DataPaths.data / 'cache'
for p in sorted(cache_dir.rglob('*.parquet')):
    rel = p.relative_to(cache_dir)
    size_mb = p.stat().st_size / 1024 / 1024
    print(f'  {rel}  ({size_mb:.1f} MB)')